In [ ]:
# Run folder: outputs go to runs/02_ablation_whisper_medium/
import os
if "REPO_ROOT" not in globals():
    REPO_ROOT = os.path.abspath("..")
RUN_DIR = os.path.join(REPO_ROOT, "runs", "02_ablation_whisper_medium")
os.makedirs(RUN_DIR, exist_ok=True)
os.chdir(RUN_DIR)

In [ ]:
# Install requirements
%pip install -r ../../requirements.txt

In [ ]:
# Imports
import os
import shutil
import re
import sys
from collections import OrderedDict

import numpy as np
import pandas as pd
import joblib
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn import BCEWithLogitsLoss
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, ConcatDataset

from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import KFold, StratifiedKFold

import torchaudio
from transformers import Wav2Vec2Model, Wav2Vec2Processor, AutoTokenizer, DebertaV2Model

import librosa
import parselmouth
from scipy.signal import find_peaks

import whisper
import glob
import matplotlib.pyplot as plt

In [ ]:
# Configuration
AD_WAV_DIR = "../../data/ADReSSo/diagnosis-train/diagnosis/train/audio/ad"
CN_WAV_DIR = "../../data/ADReSSo/diagnosis-train/diagnosis/train/audio/cn"

AD_RAW_FEATURES_CSV = "./data-extraction/audio_features_ad_raw.csv"
CN_RAW_FEATURES_CSV = "./data-extraction/audio_features_cn_raw.csv"
AD_RAW_EMBEDDINGS_CSV = "./data-extraction/audio_embeddings_ad_raw.csv"
CN_RAW_EMBEDDINGS_CSV = "./data-extraction/audio_embeddings_cn_raw.csv"

AD_TRANSCRIBED_DIR = "./transcripts/ad"
CN_TRANSCRIBED_DIR = "./transcripts/cn"

AUDIO_FEATURES_SCALER_PATH = "../../scalers/scaler_params_audio_features.pkl"
AUDIO_EMBEDDING_SCALER_PATH = "../../scalers/scaler_params_audio_emb.pkl"

AD_PROCESSED_TEXT_DIR = "./data-processing/processed_data/ad"
CN_PROCESSED_TEXT_DIR = "./data-processing/processed_data/cn"

AD_PROCESSED_FEATURES_DIR = "./preprocessing/processed_data/ad"
CN_PROCESSED_FEATURES_DIR = "./preprocessing/processed_data/cn"
AD_CSV = os.path.join(AD_PROCESSED_FEATURES_DIR, os.path.basename(AD_RAW_FEATURES_CSV))
CN_CSV = os.path.join(CN_PROCESSED_FEATURES_DIR, os.path.basename(CN_RAW_FEATURES_CSV))

AD_EMBEDDING_CSV = "./preprocessing/processed_data/ad/audio_embeddings_ad.csv"
CN_EMBEDDING_CSV = "./preprocessing/processed_data/cn/audio_embeddings_cn.csv"

AD_TEXT_DIR = AD_PROCESSED_TEXT_DIR
CN_TEXT_DIR = CN_PROCESSED_TEXT_DIR

WHISPER_MODEL_SIZE = "medium"

TEXT_EMBEDDING_MODEL = "microsoft/deberta-v3-base"
NUM_MFCC_FEATURES = 47
NUM_EMBEDDING_FEATURES = 768
AUDIO_CHANNELS = 1
CUDA = True

BATCH_SIZE = 8
EPOCHS = 200
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-4
NUM_FOLDS = 5
SAVE_BEST_MODEL = True
EARLY_STOPPING_PATIENCE = 10

SAVE_MODEL_PATH = "./results/model"
LOG_PATH = "./results/training.log"
os.makedirs(os.path.dirname(SAVE_MODEL_PATH) or ".", exist_ok=True)
os.makedirs(os.path.dirname(LOG_PATH) or ".", exist_ok=True)

TEST_WAV_DIR = "../../data/ADReSSo/diagnosis-test/diagnosis/test-dist/audio"

TEST_RAW_FEATURES_CSV = "./data-extraction/audio_features_test_raw.csv"
TEST_RAW_EMBEDDINGS_CSV = "./data-extraction/audio_embeddings_test_raw.csv"
TEST_TRANSCRIBED_DIR = "./transcripts/test"

TEST_PROCESSED_TEXT_DIR = "./data-processing/processed_data/test"
TEST_PROCESSED_FEATURES_DIR = "./preprocessing/processed_data/test"
TEST_CSV = os.path.join(TEST_PROCESSED_FEATURES_DIR, os.path.basename(TEST_RAW_FEATURES_CSV))
TEST_EMBEDDING_CSV = "./preprocessing/processed_data/test/audio_embeddings_test.csv"

TEST_LABELS_PATH = "../../data/ADReSSo/diagnosis-test/diagnosis/test-dist/task1.csv"
TEST_LABELS_ID_COLUMN = "ID"
TEST_LABELS_DX_COLUMN = "Dx"
TEST_LABELS_AD_VALUE = "ProbableAD"

INFERENCE_MODEL_PATH = "./results/model_best.pth"
INFERENCE_TEXT_FILE_PATH = "./inference/patient_processed.txt"
INFERENCE_EMBEDDING_CSV = "./inference/patient_embeddings.csv"
INFERENCE_AUDIO_FEATURES_CSV = "./inference/patient_features.csv"

In [ ]:
# Reproducibility
import random

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [ ]:
# Transcription
def transcribe_audio_files(data_path, model_size="base", output_base=None):
    model = whisper.load_model(model_size)

    output_base = output_base or os.path.join(os.path.dirname(data_path), 'extracted_data')

    for root, dirs, files in os.walk(data_path):
        for file in files:
            if file.lower().endswith('.wav'):
                audio_path = os.path.join(root, file)
                relative_path = os.path.relpath(root, data_path)
                output_dir = os.path.join(output_base, relative_path)
                os.makedirs(output_dir, exist_ok=True)
                output_file = os.path.join(output_dir, os.path.splitext(file)[0] + '.txt')
                print(f"Transcribing {audio_path}...")
                result = model.transcribe(audio_path)
                text = result["text"]
                with open(output_file, 'w', encoding='utf-8') as f:
                    f.write(text)
                print(f"Saved transcription to {output_file}")

In [ ]:
# Acoustic features
def extract_features(audio_path, sr=22050, frame_length=2048, hop_length=512, silence_threshold=0.1):
    y, sr = librosa.load(audio_path, sr=sr)
    duration = librosa.get_duration(y=y, sr=sr)
    energy = np.array([
        sum(abs(y[i:i+frame_length]**2))
        for i in range(0, len(y), hop_length)
    ])
    threshold = np.percentile(energy, silence_threshold * 100)
    silence_frames = energy < threshold
    speech_frames = ~silence_frames
    total_speech_frames = np.sum(speech_frames)
    total_speech_time = total_speech_frames * hop_length / sr
    silence_durations = []
    current_silence = 0
    for is_silence in silence_frames:
        if is_silence:
            current_silence += 1
        elif current_silence > 0:
            silence_durations.append(current_silence * hop_length / sr)
            current_silence = 0
    num_pauses = len(silence_durations)
    total_pause_duration = sum(silence_durations)
    avg_pause_duration = np.mean(silence_durations) if silence_durations else 0
    max_pause_duration = max(silence_durations) if silence_durations else 0
    pause_duration_std = np.std(silence_durations) if silence_durations else 0
    pitches, magnitudes = librosa.piptrack(y=y, sr=sr, n_fft=frame_length, hop_length=hop_length)
    pitches = pitches.flatten()
    magnitudes = magnitudes.flatten()
    pitches = pitches[magnitudes > np.median(magnitudes)]
    pitch_mean = np.mean(pitches) if len(pitches) > 0 else 0
    pitch_std = np.std(pitches) if len(pitches) > 0 else 0
    pitch_range = np.ptp(pitches) if len(pitches) > 0 else 0
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=frame_length, hop_length=hop_length)
    intensity = librosa.amplitude_to_db(S, ref=np.max)
    intensity_mean = np.mean(intensity)
    intensity_std = np.std(intensity)
    intensity_range = np.ptp(intensity)

    spectral_centroids = librosa.feature.spectral_centroid(y=y, sr=sr, n_fft=frame_length, hop_length=hop_length)[0]
    spectral_centroid_mean = np.mean(spectral_centroids)
    spectral_centroid_std = np.std(spectral_centroids)

    zcr = librosa.feature.zero_crossing_rate(y, frame_length=frame_length, hop_length=hop_length)[0]
    zcr_mean = np.mean(zcr)
    zcr_std = np.std(zcr)

    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13, n_fft=frame_length, hop_length=hop_length)
    mfccs_mean = np.mean(mfccs, axis=1)
    mfccs_std = np.std(mfccs, axis=1)

    articulation_rate = total_speech_time / (duration - total_pause_duration) if (duration - total_pause_duration) > 0 else 0

    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    peaks = find_peaks(onset_env, height=np.mean(onset_env))[0]
    estimated_syllables = len(peaks)
    speaking_rate = estimated_syllables / total_speech_time if total_speech_time > 0 else 0

    snd = parselmouth.Sound(audio_path)
    pitch = snd.to_pitch(time_step=hop_length/sr)
    point_process = parselmouth.praat.call([snd, pitch], "To PointProcess (cc)")
    local_jitter = parselmouth.praat.call(point_process, "Get jitter (local)", 0, 0, 1.0 / 75, 1.0 / 500, 1.3)
    local_shimmer = parselmouth.praat.call([snd, point_process], "Get shimmer (local)", 0, 0, 1.0 / 75, 1.0 / 500, 1.3, 1.6)
    harmonicity = snd.to_harmonicity_cc(time_step=hop_length/sr)
    hnr_values = harmonicity.values
    hnr_mean = np.mean(hnr_values[hnr_values != 0]) if np.any(hnr_values != 0) else 0

    formant = snd.to_formant_burg(time_step=hop_length/sr)
    f1 = []
    f2 = []
    f3 = []
    for t in np.arange(0, snd.duration, hop_length / sr):
        try:
            f1.append(formant.get_value_at_time(1, t))
            f2.append(formant.get_value_at_time(2, t))
            f3.append(formant.get_value_at_time(3, t))
        except:
            continue
    f1 = np.array(f1)
    f2 = np.array(f2)
    f3 = np.array(f3)
    formant_1_mean = np.mean(f1) if len(f1) > 0 else 0
    formant_1_std = np.std(f1) if len(f1) > 0 else 0
    formant_2_mean = np.mean(f2) if len(f2) > 0 else 0
    formant_2_std = np.std(f2) if len(f2) > 0 else 0
    formant_3_mean = np.mean(f3) if len(f3) > 0 else 0
    formant_3_std = np.std(f3) if len(f3) > 0 else 0

    features = {
        'duration': duration,
        'total_speech_time': total_speech_time,
        'speech_pause_ratio': total_speech_time / duration if duration > 0 else 0,
        'num_pauses': num_pauses,
        'total_pause_duration': total_pause_duration,
        'avg_pause_duration': avg_pause_duration,
        'max_pause_duration': max_pause_duration,
        'pause_duration_std': pause_duration_std,
        'pitch_mean': pitch_mean,
        'pitch_std': pitch_std,
        'pitch_range': pitch_range,
        'intensity_mean': intensity_mean,
        'intensity_std': intensity_std,
        'intensity_range': intensity_range,
        'articulation_rate': articulation_rate,
        'speaking_rate': speaking_rate,
        'spectral_centroid_mean': spectral_centroid_mean,
        'spectral_centroid_std': spectral_centroid_std,
        'zcr_mean': zcr_mean,
        'zcr_std': zcr_std,
        'jitter_local': local_jitter,
        'shimmer_local': local_shimmer,
        'hnr_mean': hnr_mean,
        'formant_1_mean': formant_1_mean,
        'formant_1_std': formant_1_std,
        'formant_2_mean': formant_2_mean,
        'formant_2_std': formant_2_std,
        'formant_3_mean': formant_3_mean,
        'formant_3_std': formant_3_std,
    }

    for i, (mean, std) in enumerate(zip(mfccs_mean, mfccs_std)):
        features[f'mfcc_{i+1}_mean'] = mean
        features[f'mfcc_{i+1}_std'] = std

    return features

def process_audio_feature_files(data_path, output_csv, sr, frame_length, hop_length, silence_threshold):
    all_features = []
    for root, dirs, files in os.walk(data_path):
        for file in files:
            if file.lower().endswith('.wav'):

                audio_path = os.path.join(root, file)
                patient_id = os.path.splitext(file)[0]
                print(f"Processing {audio_path}...")
                try:
                    features = extract_features(
                        audio_path,
                        sr=sr,
                        frame_length=frame_length,
                        hop_length=hop_length,
                        silence_threshold=silence_threshold
                    )
                    features['patient_id'] = patient_id
                    features['class'] = os.path.basename(root)
                    all_features.append(features)
                except Exception as e:
                    print(f"Error processing {audio_path}: {e}")
    df = pd.DataFrame(all_features)
    os.makedirs(os.path.dirname(output_csv) or ".", exist_ok=True)
    df.to_csv(output_csv, index=False)
    print(f"Features saved to {output_csv}")

In [ ]:
# Wav2Vec2 embeddings
def extract_embeddings(audio_path, model, processor, device):
    speech_array, sampling_rate = torchaudio.load(audio_path)
    if speech_array.shape[0] > 1:
        speech_array = torch.mean(speech_array, dim=0)
        print(f"Converted stereo to mono for {audio_path}")
    else:
        speech_array = speech_array.squeeze(0)

    if sampling_rate != 16000:
        resampler = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=16000)
        speech_array = resampler(speech_array)
        sampling_rate = 16000

    speech = speech_array.numpy()

    inputs = processor(speech, sampling_rate=sampling_rate, return_tensors="pt", padding=True)
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        embeddings = model(**inputs).last_hidden_state
    embedding = embeddings.mean(dim=1).squeeze().cpu().numpy()

    return embedding

def process_audio_embedding_files(data_path, output_csv, model_name='facebook/wav2vec2-base-960h'):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    processor = Wav2Vec2Processor.from_pretrained(model_name)
    model = Wav2Vec2Model.from_pretrained(model_name)
    model.to(device)
    model.eval()

    all_embeddings = []
    for root, dirs, files in os.walk(data_path):
        for file in tqdm(files, desc=f"Processing {root}"):
            if file.lower().endswith('.wav'):
                audio_path = os.path.join(root, file)
                patient_id = os.path.splitext(file)[0]
                try:
                    embedding = extract_embeddings(audio_path, model, processor, device)
                    all_embeddings.append({'patient_id': patient_id, 'embedding': embedding})
                except Exception as e:
                    print(f"Error processing {audio_path}: {e}")
                    continue

    if len(all_embeddings) == 0:
        print("No embeddings were extracted. Please check for errors.")
        return

    df = pd.DataFrame(all_embeddings)
    embedding_cols = pd.DataFrame(df['embedding'].tolist())
    embedding_cols['patient_id'] = df['patient_id']
    os.makedirs(os.path.dirname(output_csv) or ".", exist_ok=True)
    embedding_cols.to_csv(output_csv, index=False)
    print(f"Embeddings saved to {output_csv}")

In [ ]:
# Run data extraction
transcribe_audio_files(AD_WAV_DIR, model_size=WHISPER_MODEL_SIZE, output_base=AD_TRANSCRIBED_DIR)
transcribe_audio_files(CN_WAV_DIR, model_size=WHISPER_MODEL_SIZE, output_base=CN_TRANSCRIBED_DIR)

process_audio_feature_files(AD_WAV_DIR, AD_RAW_FEATURES_CSV, sr=22050, frame_length=2048, hop_length=512, silence_threshold=0.1)
process_audio_feature_files(CN_WAV_DIR, CN_RAW_FEATURES_CSV, sr=22050, frame_length=2048, hop_length=512, silence_threshold=0.1)

process_audio_embedding_files(AD_WAV_DIR, AD_RAW_EMBEDDINGS_CSV)
process_audio_embedding_files(CN_WAV_DIR, CN_RAW_EMBEDDINGS_CSV)

In [ ]:
# Text cleaning
def preprocess_text(text):
    text = text.lower()
    text = ' '.join(text.split())
    text = re.sub(r'[^a-zA-Z0-9\s.,!?]', '', text)

    return text

def process_text_files(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    for filename in os.listdir(input_dir):
        if filename.lower().endswith('.txt'):
            input_file = os.path.join(input_dir, filename)
            output_file = os.path.join(output_dir, filename)
            with open(input_file, 'r', encoding='utf-8') as f:
                text = f.read()
            processed_text = preprocess_text(text)
            with open(output_file, 'w', encoding='utf-8') as f:
                f.write(processed_text)

            print(f"Processed {filename}")

In [ ]:
# Audio feature scaling
def preprocess_csv(input_path, output_path, scaler_path):
    df = pd.read_csv(input_path)
    df = df.drop(['jitter_local', 'shimmer_local', 'formant_1_mean',
                  'formant_1_std', 'formant_2_mean', 'formant_2_std',
                  'formant_3_mean', 'formant_3_std', 'class'], axis=1)

    patient_ids = df['patient_id']
    df = df.drop(['patient_id'], axis=1)
    df = df.apply(lambda x: x.fillna(x.mean()) if x.isnull().any() else x)
    scaler = joblib.load(scaler_path)
    scaled_features = scaler.transform(df)
    df_scaled = pd.DataFrame(scaled_features, columns=df.columns)
    df_scaled['patient_id'] = patient_ids
    output_file = os.path.join(output_path, os.path.basename(input_path))
    df_scaled.to_csv(output_file, index=False)
    print(f"Processed file saved at: {output_file}")

In [ ]:
# Audio embedding scaling
def process_csv(input_csv, scaler_path, output_csv):
    df = pd.read_csv(input_csv)
    patient_ids = df['patient_id']
    features = df.drop(columns=['patient_id'])
    features = features.apply(lambda x: x.fillna(x.mean()), axis=0)
    scaler = joblib.load(scaler_path)
    standardized_features = scaler.transform(features)
    standardized_df = pd.DataFrame(standardized_features, columns=features.columns)
    standardized_df['patient_id'] = patient_ids
    standardized_df.to_csv(output_csv, index=False)
    print(f"Processed CSV saved to '{output_csv}'")

In [ ]:
# Run data processing
os.makedirs(AD_PROCESSED_TEXT_DIR, exist_ok=True)
os.makedirs(CN_PROCESSED_TEXT_DIR, exist_ok=True)
os.makedirs(AD_PROCESSED_FEATURES_DIR, exist_ok=True)
os.makedirs(CN_PROCESSED_FEATURES_DIR, exist_ok=True)
os.makedirs(os.path.dirname(AD_EMBEDDING_CSV) or ".", exist_ok=True)
os.makedirs(os.path.dirname(CN_EMBEDDING_CSV) or ".", exist_ok=True)

process_text_files(AD_TRANSCRIBED_DIR, AD_PROCESSED_TEXT_DIR)
process_text_files(CN_TRANSCRIBED_DIR, CN_PROCESSED_TEXT_DIR)

preprocess_csv(AD_RAW_FEATURES_CSV, AD_PROCESSED_FEATURES_DIR, AUDIO_FEATURES_SCALER_PATH)
preprocess_csv(CN_RAW_FEATURES_CSV, CN_PROCESSED_FEATURES_DIR, AUDIO_FEATURES_SCALER_PATH)

process_csv(AD_RAW_EMBEDDINGS_CSV, AUDIO_EMBEDDING_SCALER_PATH, AD_EMBEDDING_CSV)
process_csv(CN_RAW_EMBEDDINGS_CSV, AUDIO_EMBEDDING_SCALER_PATH, CN_EMBEDDING_CSV)

In [ ]:
# Dataset
class DementiaDataset(Dataset):
    def __init__(
        self,
        audio_csv_path,
        embedding_csv_path,
        text_dir,
        label_value=None,
        tokenizer_model=TEXT_EMBEDDING_MODEL,
        max_length=512
    ):
        self.audio_data = pd.read_csv(audio_csv_path)
        self.embedding_data = pd.read_csv(embedding_csv_path)
        self.text_dir = text_dir
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_model)
        self.max_length = max_length
        self.data = pd.merge(
            self.audio_data,
            self.embedding_data,
            on='patient_id',
            suffixes=('_audio', '_embedding')
        )

        self.label_value = label_value
        self.data['label'] = self.label_value

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        data_row = self.data.iloc[idx]
        patient_id = data_row['patient_id']
        text_file_path = os.path.join(self.text_dir, f'{patient_id}.txt')

        with open(text_file_path, 'r') as file:
            text = file.read()

        text_tokens = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        audio_feature_columns = [
            col for col in self.audio_data.columns if col != 'patient_id'
        ]
        embedding_feature_columns = [
            col for col in self.embedding_data.columns if col != 'patient_id'
        ]
        audio_features = data_row[audio_feature_columns].values.astype(float)
        embedding_features = data_row[embedding_feature_columns].values.astype(float)
        label = torch.tensor(data_row['label'], dtype=torch.float32)

        audio_tensor = torch.tensor(audio_features, dtype=torch.float32)
        embedding_tensor = torch.tensor(embedding_features, dtype=torch.float32)

        return text_tokens, audio_tensor, embedding_tensor, label

def create_full_dataset(
    ad_text_dir,
    cn_text_dir,
    ad_csv,
    cn_csv,
    ad_embedding_csv,
    cn_embedding_csv
):
    ad_dataset = DementiaDataset(
        ad_csv,
        ad_embedding_csv,
        ad_text_dir,
        label_value=1
    )
    cn_dataset = DementiaDataset(
        cn_csv,
        cn_embedding_csv,
        cn_text_dir,
        label_value=0
    )
    full_dataset = ConcatDataset([ad_dataset, cn_dataset])
    return full_dataset

In [ ]:
# Model
class NeuroXVocal(nn.Module):
    def __init__(
        self,
        num_audio_features,
        num_embedding_features,
        text_embedding_model,
        hidden_size=768
    ):
        super(NeuroXVocal, self).__init__()

        self.text_model = DebertaV2Model.from_pretrained(text_embedding_model)
        self.hidden_size = self.text_model.config.hidden_size
        self.audio_fc = nn.Sequential(
            nn.Linear(num_audio_features, self.hidden_size),
            nn.LayerNorm(self.hidden_size),
            nn.Dropout(0.3),
            nn.ReLU()
        )

        self.embedding_fc = nn.Sequential(
            nn.Linear(num_embedding_features, self.hidden_size),
            nn.LayerNorm(self.hidden_size),
            nn.Dropout(0.3),
            nn.ReLU()
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.hidden_size,
            nhead=8,
            dropout=0.35,
            activation='gelu'
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        self.classifier = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(0.45),
            nn.Linear(self.hidden_size // 2, 1)
        )

    def forward(self, text_input, audio_input, embedding_input):
        input_ids = text_input['input_ids'].squeeze(1)
        attention_mask = text_input['attention_mask'].squeeze(1)
        text_outputs = self.text_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        text_embeddings = text_outputs.last_hidden_state
        audio_embeddings = self.audio_fc(audio_input)
        audio_embeddings = audio_embeddings.unsqueeze(1)
        embedding_embeddings = self.embedding_fc(embedding_input)
        embedding_embeddings = embedding_embeddings.unsqueeze(1)
        combined_embeddings = torch.cat(
            (audio_embeddings, embedding_embeddings, text_embeddings),
            dim=1
        )
        combined_embeddings = combined_embeddings.permute(1, 0, 2)
        transformer_output = self.transformer_encoder(combined_embeddings)
        pooled_output = transformer_output[0]
        logits = self.classifier(pooled_output).squeeze(-1)

        return logits

    def reset_parameters(self):
        def reset_layer_parameters(layer):
            if hasattr(layer, 'reset_parameters'):
                layer.reset_parameters()
        self.audio_fc.apply(reset_layer_parameters)
        self.embedding_fc.apply(reset_layer_parameters)
        self.transformer_encoder.apply(reset_layer_parameters)
        self.classifier.apply(reset_layer_parameters)

In [ ]:
# Training loop
def train_model(
    build_model,
    full_dataset,
    epochs,
    learning_rate,
    log_path,
    save_model_path,
    device,
    num_folds=5,
    save_best_model=False,
    labels=None
):
    criterion = BCEWithLogitsLoss()

    if labels is not None:
        labels = np.array(labels)
        kfold = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=SEED)
        split_iter = kfold.split(np.zeros(len(labels)), labels)
    else:
        kfold = KFold(n_splits=num_folds, shuffle=True, random_state=SEED)
        split_iter = kfold.split(full_dataset)

    history = {}
    overall_best_val_loss = float('inf')
    overall_best_fold = None

    for fold, (train_indices, val_indices) in enumerate(split_iter):
        print(f'Fold {fold+1}/{num_folds}')

        if labels is not None:
            train_counts = np.bincount(labels[train_indices].astype(int), minlength=2)
            val_counts = np.bincount(labels[val_indices].astype(int), minlength=2)
            print(f"Fold {fold+1}: train CN/AD = {train_counts[0]}/{train_counts[1]}, val CN/AD = {val_counts[0]}/{val_counts[1]}")

        train_subset = torch.utils.data.Subset(full_dataset, train_indices)
        val_subset = torch.utils.data.Subset(full_dataset, val_indices)

        train_loader = torch.utils.data.DataLoader(
            train_subset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            generator=torch.Generator().manual_seed(SEED)
        )
        val_loader = torch.utils.data.DataLoader(
            val_subset,
            batch_size=BATCH_SIZE,
            shuffle=False
        )

        model = build_model()
        if torch.cuda.device_count() > 1:
            model = nn.DataParallel(model)
        model.to(device)

        optimizer = optim.Adam(
            model.parameters(),
            lr=learning_rate,
            weight_decay=1e-4
        )
        scheduler = ReduceLROnPlateau(
            optimizer,
            mode='min',
            factor=0.5,
            patience=5
        )

        best_val_loss = float('inf')
        epochs_no_improve = 0
        history[fold] = {'train_loss': [], 'val_loss': []}

        for epoch in range(epochs):
            model.train()
            running_loss = 0.0
            all_train_outputs = []
            all_train_labels = []

            for (
                text_data,
                audio_data,
                embedding_data,
                label
            ) in tqdm(
                train_loader,
                desc=f"Fold {fold+1}, Epoch {epoch+1}/{epochs} - Training"
            ):
                optimizer.zero_grad()
                text_data = {
                    key: value.to(device) for key, value in text_data.items()
                }
                audio_data = audio_data.to(device)
                embedding_data = embedding_data.to(device)
                label = label.to(device)

                outputs = model(text_data, audio_data, embedding_data)

                loss = criterion(outputs, label.float())
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

                running_loss += loss.item()
                all_train_outputs.extend(outputs.detach().cpu().numpy())
                all_train_labels.extend(label.cpu().numpy())

            train_loss = running_loss / len(train_loader)
            train_predictions = (np.array(all_train_outputs) > 0).astype(int)
            train_accuracy = accuracy_score(
                np.array(all_train_labels),
                train_predictions
            )
            train_report = classification_report(
                np.array(all_train_labels),
                train_predictions,
                target_names=['Control', 'ProbableAD'],
                digits=4
            )

            model.eval()
            val_running_loss = 0.0
            all_val_outputs = []
            all_val_labels = []
            with torch.no_grad():
                for (
                    text_data,
                    audio_data,
                    embedding_data,
                    label
                ) in tqdm(
                    val_loader,
                    desc=f"Fold {fold+1}, Epoch {epoch+1}/{epochs} - Validation"
                ):
                    text_data = {
                        key: value.to(device) for key, value in text_data.items()
                    }
                    audio_data = audio_data.to(device)
                    embedding_data = embedding_data.to(device)
                    label = label.to(device)

                    outputs = model(text_data, audio_data, embedding_data)
                    loss = criterion(outputs, label.float())

                    val_running_loss += loss.item()
                    all_val_outputs.extend(outputs.detach().cpu().numpy())
                    all_val_labels.extend(label.cpu().numpy())

            val_loss = val_running_loss / len(val_loader)
            val_predictions = (np.array(all_val_outputs) > 0).astype(int)
            val_accuracy = accuracy_score(
                np.array(all_val_labels),
                val_predictions
            )
            val_report = classification_report(
                np.array(all_val_labels),
                val_predictions,
                target_names=['Control', 'ProbableAD'],
                digits=4
            )

            history[fold]['train_loss'].append(train_loss)
            history[fold]['val_loss'].append(val_loss)

            scheduler.step(val_loss)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                epochs_no_improve = 0
                if save_best_model:
                    best_model_path = f"{save_model_path}_fold{fold+1}_best.pth"
                    if isinstance(model, torch.nn.DataParallel):
                        torch.save(model.module.state_dict(), best_model_path)
                    else:
                        torch.save(model.state_dict(), best_model_path)
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
                    print(f"Early stopping at epoch {epoch+1} for fold {fold+1}")
                    break

            with open(log_path, 'a') as f:
                f.write(
                    f"Fold {fold+1}, Epoch {epoch+1}, "
                    f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}\n"
                )
                f.write("Train Classification Report:\n")
                f.write(f"{train_report}\n")
                f.write("Validation Classification Report:\n")
                f.write(f"{val_report}\n")

            print(
                f"Fold {fold+1}, Epoch {epoch+1}, "
                f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}"
            )
            print("Train Classification Report:")
            print(train_report)
            print("Validation Classification Report:")
            print(val_report)

        if best_val_loss < overall_best_val_loss:
            overall_best_val_loss = best_val_loss
            overall_best_fold = fold

        del model, optimizer, scheduler
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if save_best_model and overall_best_fold is not None:
        best_fold_checkpoint = f"{save_model_path}_fold{overall_best_fold+1}_best.pth"
        overall_best_checkpoint = f"{save_model_path}_best.pth"
        shutil.copy2(best_fold_checkpoint, overall_best_checkpoint)
        print(f"Overall best fold: {overall_best_fold+1} (val loss {overall_best_val_loss:.4f}) -> saved as {overall_best_checkpoint}")

    return history

In [ ]:
# Run training
print("Starting k-fold cross-validation training script...")
with open(LOG_PATH, 'w') as f:
    pass

full_dataset = create_full_dataset(
    AD_TEXT_DIR,
    CN_TEXT_DIR,
    AD_CSV,
    CN_CSV,
    AD_EMBEDDING_CSV,
    CN_EMBEDDING_CSV
)

device = torch.device('cuda' if torch.cuda.is_available() and CUDA else 'cpu')

def build_model():
    return NeuroXVocal(
        num_audio_features=NUM_MFCC_FEATURES,
        num_embedding_features=NUM_EMBEDDING_FEATURES,
        text_embedding_model=TEXT_EMBEDDING_MODEL
    )

labels = []
for ds in full_dataset.datasets:
    labels.extend(ds.data['label'].tolist())

history = train_model(
    build_model,
    full_dataset,
    EPOCHS,
    LEARNING_RATE,
    LOG_PATH,
    SAVE_MODEL_PATH,
    device,
    NUM_FOLDS,
    SAVE_BEST_MODEL,
    labels=labels
)

In [ ]:
# Plot training and validation loss
n_folds = len(history)
fig, axes = plt.subplots(1, n_folds, figsize=(5 * n_folds, 4), sharey=True)
if n_folds == 1:
    axes = [axes]

for ax, fold in zip(axes, sorted(history.keys())):
    epochs_range = range(1, len(history[fold]['train_loss']) + 1)
    ax.plot(epochs_range, history[fold]['train_loss'], label='Train Loss')
    ax.plot(epochs_range, history[fold]['val_loss'], label='Val Loss')
    ax.set_title(f'Fold {fold + 1}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
per_fold_plot_path = os.path.join(os.path.dirname(SAVE_MODEL_PATH) or ".", "loss_curves_per_fold.png")
plt.savefig(per_fold_plot_path)
plt.show()
print(f"Saved plot to {per_fold_plot_path}")

In [ ]:
# Plot training and validation loss per fold
plt.figure(figsize=(8, 5))
for fold in sorted(history.keys()):
    epochs_range = range(1, len(history[fold]['train_loss']) + 1)
    plt.plot(epochs_range, history[fold]['train_loss'], linestyle='-', label=f'Fold {fold + 1} Train')
    plt.plot(epochs_range, history[fold]['val_loss'], linestyle='--', label=f'Fold {fold + 1} Val')

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss (All Folds)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
overview_plot_path = os.path.join(os.path.dirname(SAVE_MODEL_PATH) or ".", "loss_curves_all_folds.png")
plt.savefig(overview_plot_path)
plt.show()
print(f"Saved plot to {overview_plot_path}")

In [ ]:
# Inference
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained(TEXT_EMBEDDING_MODEL)

inference_model = NeuroXVocal(
    num_audio_features=NUM_MFCC_FEATURES,
    num_embedding_features=NUM_EMBEDDING_FEATURES,
    text_embedding_model=TEXT_EMBEDDING_MODEL,
)

inference_model.to(device)
state_dict = torch.load(INFERENCE_MODEL_PATH, map_location=device)
if 'module.' in list(state_dict.keys())[0]:
    new_state_dict = OrderedDict()
    for k, v in state_dict.items():
        name = k.replace('module.', '')
        new_state_dict[name] = v
    inference_model.load_state_dict(new_state_dict)
else:
    inference_model.load_state_dict(state_dict)

inference_model.eval()
with open(INFERENCE_TEXT_FILE_PATH, 'r') as file:
    text = file.read()
text_tokens = tokenizer(
    text,
    padding='max_length',
    truncation=True,
    max_length=512,
    return_tensors='pt',
)
text_tokens = {key: value.to(device) for key, value in text_tokens.items()}
audio_features_df = pd.read_csv(INFERENCE_AUDIO_FEATURES_CSV)
audio_features = audio_features_df.drop(columns=['patient_id']).iloc[0].values.astype(float)
audio_tensor = torch.tensor(audio_features, dtype=torch.float32).unsqueeze(0).to(device)
embedding_features_df = pd.read_csv(INFERENCE_EMBEDDING_CSV)
embedding_features = embedding_features_df.drop(columns=['patient_id']).iloc[0].values.astype(float)
embedding_tensor = torch.tensor(embedding_features, dtype=torch.float32).unsqueeze(0).to(device)
with torch.no_grad():
    outputs = inference_model(text_tokens, audio_tensor, embedding_tensor)
    probabilities = torch.sigmoid(outputs)
    confidence_score = probabilities.item()
    predicted_class = 1 if confidence_score > 0.5 else 0

print(f"Predicted Class: {predicted_class}")
print(f"Confidence Score: {confidence_score:.4f}")

In [ ]:
# Extract & process the test set
transcribe_audio_files(TEST_WAV_DIR, model_size=WHISPER_MODEL_SIZE, output_base=TEST_TRANSCRIBED_DIR)

process_audio_feature_files(
    TEST_WAV_DIR, TEST_RAW_FEATURES_CSV,
    sr=22050, frame_length=2048, hop_length=512, silence_threshold=0.1
)
process_audio_embedding_files(TEST_WAV_DIR, TEST_RAW_EMBEDDINGS_CSV)

os.makedirs(TEST_PROCESSED_TEXT_DIR, exist_ok=True)
os.makedirs(TEST_PROCESSED_FEATURES_DIR, exist_ok=True)
os.makedirs(os.path.dirname(TEST_EMBEDDING_CSV) or ".", exist_ok=True)

process_text_files(TEST_TRANSCRIBED_DIR, TEST_PROCESSED_TEXT_DIR)

preprocess_csv(TEST_RAW_FEATURES_CSV, TEST_PROCESSED_FEATURES_DIR, AUDIO_FEATURES_SCALER_PATH)
process_csv(TEST_RAW_EMBEDDINGS_CSV, AUDIO_EMBEDDING_SCALER_PATH, TEST_EMBEDDING_CSV)

In [ ]:
# Predictions only (no ground-truth labels needed)
class UnlabeledPatientDataset(Dataset):
    def __init__(
        self,
        audio_csv_path,
        embedding_csv_path,
        text_dir,
        tokenizer_model=TEXT_EMBEDDING_MODEL,
        max_length=512
    ):
        self.audio_data = pd.read_csv(audio_csv_path)
        self.embedding_data = pd.read_csv(embedding_csv_path)
        self.text_dir = text_dir
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_model)
        self.max_length = max_length

        self.data = pd.merge(
            self.audio_data,
            self.embedding_data,
            on='patient_id',
            suffixes=('_audio', '_embedding')
        )

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        data_row = self.data.iloc[idx]
        patient_id = data_row['patient_id']
        text_file_path = os.path.join(self.text_dir, f'{patient_id}.txt')

        with open(text_file_path, 'r') as file:
            text = file.read()

        text_tokens = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        audio_feature_columns = [
            col for col in self.audio_data.columns if col != 'patient_id'
        ]
        embedding_feature_columns = [
            col for col in self.embedding_data.columns if col != 'patient_id'
        ]
        audio_features = data_row[audio_feature_columns].values.astype(float)
        embedding_features = data_row[embedding_feature_columns].values.astype(float)

        audio_tensor = torch.tensor(audio_features, dtype=torch.float32)
        embedding_tensor = torch.tensor(embedding_features, dtype=torch.float32)

        return text_tokens, audio_tensor, embedding_tensor, patient_id

def predict_on_test_set(prediction_dataset, model_glob_pattern, batch_size=8, output_csv=None):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    predict_loader = torch.utils.data.DataLoader(prediction_dataset, batch_size=batch_size, shuffle=False)

    checkpoint_paths = sorted(glob.glob(model_glob_pattern))
    if not checkpoint_paths:
        raise FileNotFoundError(f"No checkpoints matched: {model_glob_pattern}")
    print(f"Ensembling {len(checkpoint_paths)} checkpoints: {checkpoint_paths}")

    all_fold_probs = []
    all_patient_ids = None

    for checkpoint_path in checkpoint_paths:
        model = NeuroXVocal(
            num_audio_features=NUM_MFCC_FEATURES,
            num_embedding_features=NUM_EMBEDDING_FEATURES,
            text_embedding_model=TEXT_EMBEDDING_MODEL,
        )
        model.to(device)
        state_dict = torch.load(checkpoint_path, map_location=device)
        if 'module.' in list(state_dict.keys())[0]:
            new_state_dict = OrderedDict()
            for k, v in state_dict.items():
                new_state_dict[k.replace('module.', '')] = v
            model.load_state_dict(new_state_dict)
        else:
            model.load_state_dict(state_dict)
        model.eval()

        fold_probs = []
        fold_ids = []
        with torch.no_grad():
            for text_data, audio_data, embedding_data, patient_id in predict_loader:
                text_data = {key: value.to(device) for key, value in text_data.items()}
                audio_data = audio_data.to(device)
                embedding_data = embedding_data.to(device)

                logits = model(text_data, audio_data, embedding_data)
                probs = torch.sigmoid(logits).cpu().numpy()
                fold_probs.extend(probs.tolist())
                fold_ids.extend(list(patient_id))

        all_fold_probs.append(fold_probs)
        if all_patient_ids is None:
            all_patient_ids = fold_ids

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    mean_probs = np.mean(np.array(all_fold_probs), axis=0)
    predictions = (mean_probs > 0.5).astype(int)

    results_df = pd.DataFrame({
        'patient_id': all_patient_ids,
        'predicted_probability': mean_probs,
        'predicted_label': predictions,
        'predicted_class': ['ProbableAD' if p == 1 else 'Control' for p in predictions],
    })
    print(f"Predicted {(results_df['predicted_label'] == 1).sum()} ProbableAD / "
          f"{(results_df['predicted_label'] == 0).sum()} Control out of {len(results_df)} recordings.")

    if output_csv is not None:
        os.makedirs(os.path.dirname(output_csv) or ".", exist_ok=True)
        results_df.to_csv(output_csv, index=False)
        print(f"Saved predictions to {output_csv}")

    return results_df

prediction_dataset = UnlabeledPatientDataset(
    TEST_CSV,
    TEST_EMBEDDING_CSV,
    TEST_PROCESSED_TEXT_DIR,
)

predictions_df = predict_on_test_set(
    prediction_dataset,
    model_glob_pattern=f"{SAVE_MODEL_PATH}_fold*_best.pth",
    output_csv=os.path.join(os.path.dirname(SAVE_MODEL_PATH) or ".", "test_predictions.csv")
)
predictions_df

In [ ]:
# Test dataset with labels (optional)
class TestPatientDataset(Dataset):
    def __init__(
        self,
        audio_csv_path,
        embedding_csv_path,
        text_dir,
        labels_df,
        id_column,
        dx_column,
        ad_value,
        tokenizer_model=TEXT_EMBEDDING_MODEL,
        max_length=512
    ):
        self.audio_data = pd.read_csv(audio_csv_path)
        self.embedding_data = pd.read_csv(embedding_csv_path)
        self.text_dir = text_dir
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_model)
        self.max_length = max_length

        self.data = pd.merge(
            self.audio_data,
            self.embedding_data,
            on='patient_id',
            suffixes=('_audio', '_embedding')
        )

        label_lookup = dict(zip(labels_df[id_column], labels_df[dx_column]))
        missing = self.data[~self.data['patient_id'].isin(label_lookup)]
        if len(missing) > 0:
            print(f"Warning: {len(missing)} test patients have no matching ground-truth "
                  f"label and will be excluded: {missing['patient_id'].tolist()}")
            self.data = self.data[self.data['patient_id'].isin(label_lookup)].reset_index(drop=True)

        self.data['label'] = self.data['patient_id'].map(
            lambda pid: 1 if label_lookup[pid] == ad_value else 0
        )

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        data_row = self.data.iloc[idx]
        patient_id = data_row['patient_id']
        text_file_path = os.path.join(self.text_dir, f'{patient_id}.txt')

        with open(text_file_path, 'r') as file:
            text = file.read()

        text_tokens = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        audio_feature_columns = [
            col for col in self.audio_data.columns if col != 'patient_id'
        ]
        embedding_feature_columns = [
            col for col in self.embedding_data.columns if col != 'patient_id'
        ]
        audio_features = data_row[audio_feature_columns].values.astype(float)
        embedding_features = data_row[embedding_feature_columns].values.astype(float)
        label = torch.tensor(data_row['label'], dtype=torch.float32)

        audio_tensor = torch.tensor(audio_features, dtype=torch.float32)
        embedding_tensor = torch.tensor(embedding_features, dtype=torch.float32)

        return text_tokens, audio_tensor, embedding_tensor, label, patient_id

In [ ]:
# Ensemble evaluation across all trained folds
def evaluate_on_test_set(test_dataset, model_glob_pattern, batch_size=8, output_dir=None):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    checkpoint_paths = sorted(glob.glob(model_glob_pattern))
    if not checkpoint_paths:
        raise FileNotFoundError(f"No checkpoints matched: {model_glob_pattern}")
    print(f"Ensembling {len(checkpoint_paths)} checkpoints: {checkpoint_paths}")

    all_fold_probs = []
    all_labels = None
    all_patient_ids = None

    for checkpoint_path in checkpoint_paths:
        model = NeuroXVocal(
            num_audio_features=NUM_MFCC_FEATURES,
            num_embedding_features=NUM_EMBEDDING_FEATURES,
            text_embedding_model=TEXT_EMBEDDING_MODEL,
        )
        model.to(device)
        state_dict = torch.load(checkpoint_path, map_location=device)
        if 'module.' in list(state_dict.keys())[0]:
            new_state_dict = OrderedDict()
            for k, v in state_dict.items():
                new_state_dict[k.replace('module.', '')] = v
            model.load_state_dict(new_state_dict)
        else:
            model.load_state_dict(state_dict)
        model.eval()

        fold_probs = []
        fold_labels = []
        fold_ids = []
        with torch.no_grad():
            for text_data, audio_data, embedding_data, label, patient_id in test_loader:
                text_data = {key: value.to(device) for key, value in text_data.items()}
                audio_data = audio_data.to(device)
                embedding_data = embedding_data.to(device)

                logits = model(text_data, audio_data, embedding_data)
                probs = torch.sigmoid(logits).cpu().numpy()
                fold_probs.extend(probs.tolist())
                fold_labels.extend(label.numpy().tolist())
                fold_ids.extend(list(patient_id))

        all_fold_probs.append(fold_probs)
        if all_labels is None:
            all_labels = fold_labels
            all_patient_ids = fold_ids

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    mean_probs = np.mean(np.array(all_fold_probs), axis=0)
    predictions = (mean_probs > 0.5).astype(int)
    true_labels = np.array(all_labels)

    test_accuracy = accuracy_score(true_labels, predictions)
    test_report = classification_report(
        true_labels, predictions, target_names=['Control', 'ProbableAD'], digits=4
    )

    print(f"Independent test accuracy (ensemble of {len(checkpoint_paths)} folds): {test_accuracy:.4f}")
    print(test_report)

    results_df = pd.DataFrame({
        'patient_id': all_patient_ids,
        'true_label': true_labels,
        'predicted_probability': mean_probs,
        'predicted_label': predictions,
    })

    if output_dir is not None:
        os.makedirs(output_dir, exist_ok=True)
        results_csv_path = os.path.join(output_dir, "test_evaluation.csv")
        report_path = os.path.join(output_dir, "test_report.txt")
        results_df.to_csv(results_csv_path, index=False)
        with open(report_path, 'w') as f:
            f.write(f"Independent test accuracy (ensemble of {len(checkpoint_paths)} folds): {test_accuracy:.4f}\n\n")
            f.write(test_report)
        print(f"Saved per-patient results to {results_csv_path}")
        print(f"Saved accuracy/classification report to {report_path}")

    return test_accuracy, test_report, results_df

test_labels_df = pd.read_csv(TEST_LABELS_PATH)
test_dataset = TestPatientDataset(
    TEST_CSV,
    TEST_EMBEDDING_CSV,
    TEST_PROCESSED_TEXT_DIR,
    labels_df=test_labels_df,
    id_column=TEST_LABELS_ID_COLUMN,
    dx_column=TEST_LABELS_DX_COLUMN,
    ad_value=TEST_LABELS_AD_VALUE
)

test_accuracy, test_report, test_results_df = evaluate_on_test_set(
    test_dataset,
    model_glob_pattern=f"{SAVE_MODEL_PATH}_fold*_best.pth",
    output_dir=os.path.dirname(SAVE_MODEL_PATH) or "."
)
test_results_df